#Brain Activity Monitoring using AI and Predictive Analysis!

This notebook implements an event detection and future prediction pipeline. The raw EEG recordings first are prperocessed by selecting EEG channels, applying filtering, resampling, and segmenting the singles into 5-secs windows. Rather than using provided labels, a frequency-based heuristic is applied to generate labels using high-frequency (20-44Hz)activities. A CNN model is then trained to detect micro-events, followed by another step to explore the prediction of future events based on the squential EEG windows.

## HOW TO USE THIS NOTEBOOK

OPTION 1 (Recommended - Fast):
- Download preprocessed dataset (event_X.npy, event_y.npy)
- Place files correctly in the path
- Run from "Load Event Data" step 4

 OPTION 2 (Full Reproduction):
- Download raw EDF dataset
- Update dataset_path
- Run preprocessing steps (Step 2 & Step 3)



# Set-up and Import

In [ ]:
#Step 1 - install+import libraries

!pip install mne
import mne
import numpy as np
import os
import gc

from google.colab import drive
drive.mount('/content/drive')

dataset_path = "/content/drive/MyDrive/FYP-Brain activity monitoring using AI and predictive analytics/EEG-Dataset/Events_Data"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Basic Preprocessing

In [ ]:
# Step 2: Basic Preprocessing (Memory-optimised Pipeline)

from scipy.signal import welch

# Path to the raw EVENT EEG dataset (edf files)
# - This is ONLY required if you want to run preprocessing from scratch
# - not required for evaluation, as preprocessed files (event_X.npy, event_y.npy) are provided
# if running locally, up date this path with your dataset location

dataset_path = "/content/drive/MyDrive/FYP-Brain activity monitoring using AI and predictive analytics/EEG-Dataset/Events_Data"

# 1. Discover all EDF file
# Recursively scan the dataset folder and collects all EEG files
edf_files = []
for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.endswith(".edf"):
            edf_files.append(os.path.join(root, file))

print(f"Total EDF files found: {len(edf_files)}")

# 2. Create output directory
# This directory stores processed batches to avoid keeping everything in RAM
# Each file will be saved separately (memory-efficient design)

temp_dir = "/content/drive/MyDrive/FYP-Brain activity monitoring using AI and predictive analytics/EEG-Dataset/Processed_Event_Data"
os.makedirs(temp_dir, exist_ok=True)
print(f"Temporary directory verified at: {temp_dir}")

# 3. Function to process one EEG file
def process_single_file(file_path):
    try:
      #load raw EEG file
      raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
      raw.pick('eeg')   #keep only EEG channels & remove non-EEG signals
      raw.set_eeg_reference(verbose=False) #set reference to reduce noise

      #bandpass filter 1-40Hz to remove noise & irrelevant frequencies
      raw.filter(1, 40, verbose=False)
      #downsample to 200Hz for consistency acros all files
      raw.resample(200, verbose=False)

      #Segement signal into fix window
      # Convert continuous EEG into 5-second segement (epchos)

      '''
        after preprocessing, each EEG recording is segmented into fixed-length windows:
      '''
      epochs = mne.make_fixed_length_epochs(raw, duration=5, preload=True, verbose=False)

      #Shape : windows, channels, time)
      X_batch = epochs.get_data(verbose=False)

      '''This step extracts frequency-based features from EEG windows and generates labels automatically using heuristic approch (welch's method).
      High-freq power (20-40Hz)is computed for each window, as elevated activity in this range may indicated abnormal events.
      A threshold is then applied to identify the top 10% of high-frequency segments, which are labelled as micro-events, and the remaining segments are labelled as normal events.
      '''
      # compute high-frequency power 20-40 hz for each EEG window
      powers = []
      for epoch in X_batch:
        f, psd = welch(epoch, fs=200, nperseg=256)
        high_freq_mask = (f >= 20) & (f <= 40) #Select high-frq band

        #average power across channels in selected frequencies band
        avg_power = psd[:, high_freq_mask].mean()
        powers.append(avg_power)

      # convert list to NumPy array
      powers = np.array(powers)

      # define threshold using top 10% highest activity
      thresh = np.percentile(powers, 90)

      # assign labels 1 = micro-event (high activity, 0 = normal activity
      y_batch = (powers > thresh).astype(int)

      return X_batch, y_batch

    except Exception as e:
    #if a file fails, skip it safely
      print(f"Error processing {file_path}: {e}")
      return None, None

# 4. Process all files (memory-safe loop)
# Instead of loading all data in RAM, each file is processed and saved individually
file_limit = len(edf_files)
print(f"Processing {file_limit} files. Saving individually to Drive to prevent RAM crash...")

for i in range(file_limit):
  #Defind ouput file path
  save_path_x = os.path.join(temp_dir, f"X_{i}.npy")

  #skip if already processed
  if os.path.exists(save_path_x):
      continue

  #process single file
  x_d, y_d = process_single_file(edf_files[i])

  #save only valid results
  if x_d is not None:
    #use float32 to reduce storage and memory usage
    np.save(save_path_x, x_d.astype(np.float32))
    #label stored as int 8 - effiecient for binary classification
    np.save(os.path.join(temp_dir, f"y_{i}.npy"), y_d.astype(np.int8))

  # Free memory after each file to prevent RAM overflow
  del x_d, y_d

  #print progress
  if i % 10 == 0:
    print(f"Processed {i} of {file_limit}...")
    gc.collect()

print("Processing complete! All files saved as batches in Drive.")

Total EDF files found: 1069
Temporary directory verified at: /content/drive/MyDrive/FYP-Brain activity monitoring using AI and predictive analytics/EEG-Dataset/Processed_Event_Data
Processing 1069 files. Saving individually to Drive to prevent RAM crash...
Error processing /content/drive/MyDrive/FYP-Brain activity monitoring using AI and predictive analytics/EEG-Dataset/Events_Data/raw_data/edf/Normal EDF Files/0000725.edf: No events produced, check the values of start, stop, and duration
Processing complete! All files saved as batches in Drive.


## This step merges all processed EEG window files into one large dataset:(OPTIONAL)
- event_X.npy (EEG data)
- event_y.npy (labels)

 This step was already executed during development.
 The final merged dataset is PROVIDED and can be directly loaded.

 This step is memory-intensive and time-consuming.
 It should ONLY be run if reproducing the full preprocessing pipeline.

 Note: You can skip this step and go directly to data loading in step 4.

In [ ]:
# # Step 3: Merge all processed event files into one large dataset and saved as event_X.npy and event_y.npy to gooogle drive

# import os #
# import numpy as np

# #path where all processed files are saved from step 2
# temp_dir = "/content/drive/MyDrive/FYP-Brain activity monitoring using AI and predictive analytics/EEG-Dataset/Processed_Event_Data"

# #Final output files where the full dataset event_y.npy & event_X.npy
# output_path_X = "/content/drive/MyDrive/FYP-Brain activity monitoring using AI and predictive analytics/EEG-Dataset/event_X.npy"
# output_path_y = "/content/drive/MyDrive/FYP-Brain activity monitoring using AI and predictive analytics/EEG-Dataset/event_y.npy"

# #check if step 2 actually created the files
# if not os.path.exists(temp_dir):
#   print(f"Error: {temp_dir} does not exist. Run step 2")

# else:
#   #get all feature files ans sort them in correct order
#   all_X_files = sorted(
#       [f for f in os.listdir(temp_dir) if f.startswith('X_')],
#       key=lambda x: int(x.split('_')[1].split('.')[0])
#   )

#   #1. work out final dataset size no loading all data
#   total_windows = 0   #total number of EEG windows across all files

#   #load 1 file just to get the shape  (channels and time length)
#   sample_x = np.load(os.path.join(temp_dir, all_X_files[0]))
#   n_channels, n_times = sample_x.shape[1], sample_x.shape[2]

#   print("Calculating total size...")
#   # Loop through all filrs to count how many window we have
#   for fx in all_X_files:
#     # mmap_mode='r' : load file without using RAM
#     temp_x = np.load(os.path.join(temp_dir, fx), mmap_mode='r')
#     total_windows += temp_x.shape[0]  # add mumber of samples

#   print(f"Total windows to merge: {total_windows}")

#   # 2. Create large file directly on disk not RAM
#   # Create a big file for X
#   X_final = np.memmap(
#       output_path_X,
#       dtype='float32',  # data type - eeg value
#       mode='w+',        # write mode
#       shape=(total_windows, n_channels, n_times)
#   )

#   # Create big file for y, label 0 or 1
#   y_final = np.memmap(
#       output_path_y,
#       dtype='int8',     # small int (save space)
#       mode='w+',
#       shape=(total_windows,)
#   )

#   # 3. Merge data file by file safe for RAM
#   curr_idx = 0  # keeps track of where we are inserting data
#   print("Merging files directly to Drive (Memory Efficient)...")
#   for fx in all_X_files:
#     fy = fx.replace('X_', 'y_')   #find matching label file
#     path_x = os.path.join(temp_dir, fx)
#     path_y = os.path.join(temp_dir, fy)

#     # only process if both X and y exist
#     if os.path.exists(path_y):
#       #load small batch
#       data_x = np.load(path_x)
#       data_y = np.load(path_y)

#       count = data_x.shape[0] #how many sample in this file

#       #insert into big dataste
#       X_final[curr_idx:curr_idx+count] = data_x
#       y_final[curr_idx:curr_idx+count] = data_y

#       #move pointer forward
#       curr_idx += count

#       # Every 1000 samples saved to disk (prevents data loss)
#       if curr_idx % 1000 == 0:
#         X_final.flush()   # physically writes data to disk
#         print(f"Merged {curr_idx} windows...")

#   # finaly save everything properly
#   X_final.flush()
#   y_final.flush()

#   # Free memory / close files
#   del X_final, y_final

#   print(f"Success! Master dataset saved to Drive. Total: {total_windows} windows.")

## Loading saved files in step 3 to avoid running atep 3 again:

This loads:

 event_X.npy → EEG windows (shape: windows, channels, time)

event_y.npy → labels (0 = normal, 1 = micro-event)

 Uses memory mapping (np.memmap) to avoid loading full dataset into RAM.

In [ ]:
#Step 4: Load the processed Event data
import numpy as np
import os

#Path to the saved data in step 3
output_path_X = "/content/drive/MyDrive/FYP-Brain activity monitoring using AI and predictive analytics/EEG-Dataset/event_X.npy"
output_path_y = "/content/drive/MyDrive/FYP-Brain activity monitoring using AI and predictive analytics/EEG-Dataset/event_y.npy"

#These valuse must match how the data was saved earlier
total_windows = 265339  #number of EEG segments (windows)
n_channels = 21         #number of EEG electrodes
n_times = 1000          # number of time points per window

#Check if the dataset files exist in Google Drive
if os.path.exists(output_path_X) and os.path.exists(output_path_y):
  print("Mapping dataset from Drive via Memmap...")

  ''' Instead of loading all data into RAM as it's causing colab to crash, using memory mapping (memmap) which this loads data from disk only when needed
  '''
  X = np.memmap(
      output_path_X,
      dtype='float32',  # to match how it was saved
      mode='r',         # read-only (safe and no modification)
      shape=(total_windows, n_channels, n_times)
  )

  y = np.memmap(
      output_path_y,
      dtype='int8',     # Labels are smaller 0 or 1
      mode='r',
      shape=(total_windows,)
  )

  # to confirm everything was loaded correctly
  print("Successfully mapped!")
  print("Final Shapes:", X.shape, y.shape)

else:
  print("Error: File not found")
  print("Please ensure Step 3 finished successfully.")

Mapping dataset from Drive via Memmap...
Successfully mapped!
Final Shapes: (265339, 21, 1000) (265339,)


## Train/Test Split
A memory-efficient train/test split was created using indices with stratification to maintain class balance.

But, the indices were directly used in training, as a file-by-file loading approach.

In [ ]:
# Step 5: Memory-efficient Train/Test Split using indices

import numpy as np
import torch
import random
from sklearn.model_selection import train_test_split


# 1. Set seed for reproducibility
seed = 100
np.random.seed(seed)
torch.manual_seed(seed)
random.seed(seed)
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(seed)

# 2. Create indices for the dataset
# X and y are already loaded as np.memmap objects from Step 4
indices = np.arange(len(X))

# 3. Split the indices not the data itself, we also get the corresponding y value for stratification.
# y_stratify is just a helper for train-test_split to ensure balanced classes in splits
train_indices, test_indices, _, _= train_test_split(
    indices, y, test_size=0.2, random_state=100, stratify=y
)

print(f"Number of training samples (indices): {len(train_indices)} (Events: {np.sum(y[train_indices])})")
print(f"Number of testing samples (indices): {len(test_indices)} (Events: {np.sum(y[test_indices])})")

Number of training samples (indices): 212271 (Events: 6959)
Number of testing samples (indices): 53068 (Events: 1740)


## Create a custom PyTorch Dataset for loading EEG data file-by-file

In [ ]:
# Step 6: Loading EEG data file by file
import os
import numpy as np
import torch
from torch.utils.data import Dataset

class EEGDataset(Dataset):
  def __init__(self, folder_path, mean=None,std=None,):
    # Store folder path and normalisations values
    self.folder = folder_path
    self.files =[]  # will store valid file IDs
    self.mean = mean
    self.std = std

    #Loop thru all files in the folder
    for f in sorted(os.listdir(self.folder)):
      if f.startswith('X_'):  #only look at X files
        # Extract file ID e.g X_12.npy -> 12
        file_id = f.split('_')[1].split('.')[0]

        # Create full paths for x and y files
        x_path = os.path.join(folder_path, f)
        y_path = os.path.join(folder_path, f'y_{file_id}.npy')

        # fast check: make sure both files exist & are not empty
        if os.path.exists(x_path) and os.path.getsize(x_path) > 0 and os.path.exists(y_path) and os.path.getsize(y_path) > 0:
          self.files.append(file_id)  #store valid file id

    print(f"Valid files: {len(self.files)}")

  def __len__(self):
    # Return total num of vaild files
    return len(self.files)

  def __getitem__(self, idx):
    # get file id for this index
    file_id = self.files[idx]

    # Load X -> EEG data & y -> labels from disk
    X_data = np.load(os.path.join(self.folder, f'X_{file_id}.npy'))
    y_data = np.load(os.path.join(self.folder, f'y_{file_id}.npy'))

    # this apply normalisation if mean and std are provided
    if self.mean is not None and self.std is not None:
      X_data = (X_data - self.mean) / self.std

    # Convert to PyTorch tensors
    return torch.tensor(X_data, dtype=torch.float32), torch.tensor(y_data, dtype=torch.long)

## Building a CNN Model

In [ ]:
#Step 7: Building the CNN
import torch.nn as nn

class EEGNet(nn.Module):
  def __init__(self):
    super(EEGNet, self).__init__()

    #First(layer) look at the 21 channels to find the basic electrical patterns
    self.conv1 = nn.Conv1d(21, 32, kernel_size=7, padding=3)
    self.bn1 = nn.BatchNorm1d(32)

    #2ed layer to find more complex relationships in those patterns
    self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
    self.bn2 = nn.BatchNorm1d(64)

    #Shrinks the data by half to focus only on the strongest signals
    self.pool = nn.MaxPool1d(2)

    #Randomly ignors some connectiin to prevent the model form (overfitting) just memorising the training data
    self.dropout = nn.Dropout(0.5)

    #Final decision layer: determine if the data is normal or micro-event
    # Using the known n_times (1000) from earlier steps
    self.fc = nn.Linear(64 * (n_times // 2), 2)

  def forward(self, x):
    #Pass data through 1st kayer, activate ir (relu) and shrink it (pool)
    x = self.pool(torch.relu(self.bn1(self.conv1(x))))

    #Pass through the 2ed layer and activate
    x = torch.relu(self.bn2(self.conv2(x)))

    #apply dropout to keep the model flexible
    x = self.dropout(x)

    #Flatten the data from a 2D to 1D line so the final layer can read it
    x = x.view(x.size(0), -1)

    #Output the final prediction
    return self.fc(x)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EEGNet().to(device)
print(model)

EEGNet(
  (conv1): Conv1d(21, 32, kernel_size=(7,), stride=(1,), padding=(3,))
  (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=32000, out_features=2, bias=True)
)


## Defining loss function and optimizer

In [ ]:
# Step 7.1
import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

## Train the model using file by file loading for memory-efficient

In [ ]:
# Step 8: model training

# create dataset object to load processed files
dataset = EEGDataset(temp_dir)

for epoch in range(1): # run only 1 epoch to keep it fast
  model.train() #set model to training mode
  total_loss = 0  # track total loss for this epoch

  #Loop through dataset (limit to 500 files to avoid long runtime)
  for i in range(min(500, len(dataset))):
    # load 1 file X -> EEG data, y -> labels
    X_batch, y_batch = dataset[i]

    # move data to GPU/CPU
    X_batch = X_batch.to(device)
    y_batch = y_batch.to(device)

    # fix shape if extra dimension exists as it's commen with EEG data
    if X_batch.dim() == 4 and X_batch.shape[1] == 1:
      X_batch = X_batch.squeeze(1)


    optimizer.zero_grad() # Reset gradients from previous step
    outputs = model(X_batch)  # Forward pass (model prediction)

    # Compute loss - difference between prdiction and true labels
    loss = criterion(outputs, y_batch)

    # Backpropagation - calculate gradients
    loss.backward()

    optimizer.step()  # Update model's weights

    total_loss +=loss.item() # add loss to total

    #free memory to prevent RAM issues
    del X_batch, y_batch
    torch.cuda.empty_cache()

  # print total loss for the epoch
  print(f"Epoch {epoch + 1}, Total Loss: {total_loss:.4f} ")

Valid files: 1068
Epoch 1, Total Loss: 346.0964 


## Testing and Evaluatation the model

In [ ]:
# Step 9: Event Evaluation
from sklearn.metrics import accuracy_score, classification_report
model.eval()  # set model to evaluation mode

all_preds = []  # store all predicted labels
all_labels = []   # store all true labels

# disable gradients -> faster and uses less memory
with torch.no_grad():
  for i in range(min(500, len(dataset))):

    # load 1 file X -> EEG data, y -> labels
    X_batch, y_batch = dataset[i]

    # move data to GPU/CPU
    X_batch = X_batch.to(device)
    y_batch = y_batch.to(device)

    # fix shape if extra dimension exists as it's commen with EEG data
    if X_batch.dim() == 4 and X_batch.shape[1] == 1:
      X_batch = X_batch.squeeze(1)

    # get model predictions
    outputs = model(X_batch)

    # convert outputs to predicated class 0 or 1
    preds = torch.argmax(outputs, axis=1)

    # store predictions and true labels
    all_preds.extend(preds.cpu().numpy())
    all_labels.extend(y_batch.cpu().numpy())

# Print result
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("\nClassification Report:\n", classification_report(all_labels, all_preds, target_names=['Normal', 'Micro-Event']))

Accuracy: 0.8898321003998462

Classification Report:
               precision    recall  f1-score   support

      Normal       0.97      0.91      0.94    108318
 Micro-Event       0.06      0.16      0.09      3475

    accuracy                           0.89    111793
   macro avg       0.51      0.54      0.51    111793
weighted avg       0.94      0.89      0.91    111793



# Create lables for predictions for Future events

In [ ]:
# Step 10: Create labels for prediction (Future event)

# To predict the future, we need to shift the label y by 1 step
# This makes the model look at the current brain activity but predict what happens next
y_pred = np.roll(y, -1)

#Because we shifted everything, the very last piece of data now has no next event to point to
# removed last element from both brain data X and label y
X_pred = X[:-1]
y_pred = y_pred[:-1]


## Class imbalance distribution

In [ ]:
# step 10.1 Class imbalance
print("Class distribution (Future prediction lables:)")
print(np.bincount(y_pred))

Class distribution (Future prediction:)
[256639   8699]


## Training the model for future prediction

In [ ]:
# Step 11
from torch.utils.data import DataLoader, TensorDataset

max_samples = 20000   # using total 20000 samples

#Only select a subset of the data for training
X_pred = X_pred[:max_samples]
y_pred = y_pred[:max_samples]

# convert to NumPy arrays to PyTorch tensors
X_pred = torch.tensor(X_pred, dtype=torch.float32)
y_pred = torch.tensor(y_pred, dtype=torch.long)

# create Dataloader for batching and shuffling data during trainig
loader = DataLoader(
    TensorDataset(X_pred, y_pred),
    batch_size=64,
    shuffle=True
)

# initialise CNN  model and move to GPU/CPU
model_pred = EEGNet().to(device)

# Define optimizer (Adam) with a smaller learning rata for stable training
optimizer = torch.optim.Adam(model_pred.parameters(), lr=0.0001)

# Apply class weights to reduce impact of class imbalance
weights = torch.tensor([1.0, 5.0]).to(device)

# define function (crossEntropy) with class weighting
criterion = nn.CrossEntropyLoss(weight=weights)

# Training loop
for epoch in range(5):  #use 5 for faster execution
  model_pred.train()  #set model to training mode
  total_loss = 0  #total loss for epoch

  # loop through batches of data
  for x_batch, y_batch in loader:
    x_batch, y_batch = x_batch.to(device), y_batch.to(device) # move data to device

    optimizer.zero_grad() # clear pervious gradients
    outputs = model_pred(x_batch) # forward pass(prediction)
    loss = criterion(outputs, y_batch) # compute loss
    loss.backward() # backpropagation
    optimizer.step()  # update model weights

    total_loss += loss.item() #accumulate loss

  # print average loss per epoch
  print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")


Epoch 1, Loss: 0.4708
Epoch 2, Loss: 0.4283
Epoch 3, Loss: 0.4158
Epoch 4, Loss: 0.4042
Epoch 5, Loss: 0.3955


## Evaluate

In [ ]:
# Step 12: evaluate model's performance on future prediction

from sklearn.metrics import accuracy_score, classification_report

model_pred.eval()   # set model to evaluation mode
all_preds = []  # store all predicted labels
all_labels = []  # store all true labels

# disable gradient computation for faster and memory-efficent inference
with torch.no_grad():
  for x_batch, y_batch in loader:

    # move imput data to GPU/CPU
    x_batch = x_batch.to(device)

    # forward pass: get model prediction
    outputs = model_pred(x_batch)

    #convert outputs tp predicted class 0 or 1
    preds = torch.argmax(outputs, dim=1).cpu().numpy()

    # store prediction and corresponding true labes
    all_preds.extend(preds)
    all_labels.extend(y_batch.numpy())

# calculate and print accuracy overall
print("Future Prediction Accuracy:", accuracy_score(all_labels, all_preds))
print("\nClassification Report:\n", classification_report(all_labels, all_preds, target_names=['Normal', 'Micro-Event']))

Future Prediction Accuracy: 0.0387

Classification Report:
               precision    recall  f1-score   support

      Normal       0.00      0.00      0.00     19226
 Micro-Event       0.04      1.00      0.07       774

    accuracy                           0.04     20000
   macro avg       0.02      0.50      0.04     20000
weighted avg       0.00      0.04      0.00     20000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
